Pydantic 负责：
- 解析
- 类型转换
- 约束校验
- 创建类型明确的 Python 对象

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class UserCreateDTO(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
    )

    name: str = Field(min_length=2, max_length=50)
    age: int = Field(ge=18)
    active: bool = True

### Pydantic 默认可能进行类型转换
类型强制转换或类型解析,如果转换失败产生 ValidationError。

### 额外字段策略
模型没有声明，Pydantic 默认通常会忽略额外字段，但安全敏感的输入模型可以配置：

`model_config = ConfigDict(extra="forbid")`额外字段会导致校验失败。

- extra="ignore" → 忽略额外字段
- extra="forbid" → 拒绝额外字段
- extra="allow"  → 保留额外字段

### 序列化
Pydantic v2 转换为字典`data = user.model_dump()`
`model_dump_json`则是转换成json字符串

### model_validate()
`user = UserCreateDTO.model_validate(raw_data)` 校验字典或其他对象；也可以用`user = UserCreateDTO(**raw_data)`这种方式

### DTO 是什么？
是 Data Transfer Object，数据传输对象。
它用于定义不同系统层之间传输的数据结构。

#### DTO 与 ORM Model？
DTO 描述接口边界
ORM Model 描述数据库表和关系

#### 从对象属性构建响应 DTO
Pydantic v2 可以配置
```
class UserResponseDTO(BaseModel):
    model_config = ConfigDict(from_attributes=True)

    id: int
    name: str
    email: str

response = UserResponseDTO.model_validate(orm_user)
```
Pydantic 会从对象属性中读取字段。

#### ValidationError
错误列表会包含多个字段问题,FastAPI 会接住请求模型的 ValidationError，并通常转换成 HTTP 422。

gt → greater than               严格大于
ge → greater than or equal      大于等于
lt → less than                  严格小于
le → less than or equal         小于等于